# D-FINE annotation → YOLO11n training and pruning → local CPU inference

Start this notebook after changing to your `efficient_detection/` checkout. Files such as `yolo/train_yolo.py` and `requirements/training.txt` are accessed directly from the current directory. Attach your labeled dataset and model checkpoints as Kaggle Datasets. Enable a Kaggle GPU and Internet for installing dependencies/downloading YOLO weights. D-FINE annotation is optional when you already have labels. Pruning performs **zero recovery training**. Exports use **FP32**, without quantization or knowledge distillation.


In [ ]:
from pathlib import Path

REPO = Path.cwd().resolve()  # Already inside efficient_detection/
# Dataset: https://www.kaggle.com/datasets/flap1812/fisheye8k
# Adjust DATA_ROOT if Kaggle mounts the dataset at a different path.
DATA_ROOT = Path("/kaggle/input/fisheye8k/Fisheye8K")
DATA_FORMAT = "yolo"  # 'yolo' or 'coco'
IMAGES = DATA_ROOT / "train/images"
LABELS = DATA_ROOT / "train/labels"
ANNOTATIONS = DATA_ROOT / "train/train.json"  # COCO only
# AICITY Track 4 uses the original FishEye8K test split for validation.
VAL_IMAGES = DATA_ROOT / "test/images"  # None to split the input training set
VAL_LABELS = DATA_ROOT / "test/labels"
VAL_ANNOTATIONS = DATA_ROOT / "test/test.json"
CATEGORY_MAP = None  # Optional source COCO category-ID mapping JSON
SPLIT = "camera"  # Use 'random' only when camera IDs are unavailable

UNLABELED_IMAGES = None  # e.g. Path('/kaggle/input/extra-training/images')
ANNOTATION_MODEL = "dfine-s"  # dfine-s, dfine-m, dfine-l, dfine-x
ANNOTATION_CHECKPOINT = None  # Matching .pth checkpoint; required with unlabeled images
ANNOTATION_CHECKPOINT_CLASSES = 80  # 5 for a custom five-class checkpoint
ANNOTATION_CONFIDENCE = 0.5
YOLO_CHECKPOINT = "yolo11n.pt"  # Or an attached pretrained YOLO11n checkpoint path

WORK = REPO.parent / "yolo_workflow"
PREPARED = WORK / "prepared"
RUNS = WORK / "runs"
IMAGE_SIZE = 640
EPOCHS = 30
BATCH_SIZE = 2
EXPORT_SIZES = [512, 640, 960]
INCLUDE_VALIDATION_IN_BUNDLE = (
    True  # Real image copies for accuracy benchmarking locally
)

## 1. Install dependencies
Run from the current efficient_detection checkout. The requirements register the package in editable mode so direct script commands can resolve imports. Keep Kaggle's matched PyTorch/torchvision installation. D-FINE source is fetched only when annotation is enabled.


In [ ]:
import json
import subprocess
import sys

assert (REPO / "yolo/train_yolo.py").is_file(), "Change to efficient_detection/ first"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REPO / "requirements/training.txt"),
    ],
    cwd=REPO,
    check=True,
)
import torch

assert torch.cuda.is_available(), "Enable a GPU accelerator"


def run(script, *arguments):
    subprocess.run(
        [
            sys.executable,
            "-u",
            str(REPO / "yolo" / script),
            *map(str, arguments),
        ],
        cwd=REPO,
        check=True,
    )


if UNLABELED_IMAGES is not None:
    subprocess.run(
        [sys.executable, str(REPO / "dfine/setup_backend.py")],
        cwd=REPO,
        check=True,
    )

## 2. Prepare the human-labeled splits first
YOLO IDs must follow bus, bike, car, pedestrian, truck (0–4). COCO categories are remapped by name, or using `CATEGORY_MAP`. Existing validation splits are preserved; otherwise split by camera. Preparation precedes annotation so held-out images/cameras can be excluded.

Prepared data and completed stages are reused on reruns. Choose a new `WORK` directory when changing data, annotations, or experiment settings.

Defaults match flap1812/fisheye8k: Fisheye8K/train/{images,labels} and Fisheye8K/test/{images,labels}. The original test split is used as validation following AICITY Track 4; it is not an untouched test set for model selection. Use another WORK directory if changing datasets or splits.


In [ ]:
if not (PREPARED / "data.yaml").is_file():
    arguments = [
        "--format",
        DATA_FORMAT,
        "--images",
        IMAGES,
        "--output",
        PREPARED,
        "--split",
        SPLIT,
    ]
    if DATA_FORMAT == "yolo":
        arguments += ["--labels", LABELS]
    else:
        arguments += ["--annotations", ANNOTATIONS]
        if CATEGORY_MAP is not None:
            arguments += ["--category-map", CATEGORY_MAP]
    if VAL_IMAGES is not None:
        arguments += ["--val-images", VAL_IMAGES]
        arguments += (
            ["--val-labels", VAL_LABELS]
            if DATA_FORMAT == "yolo"
            else ["--val-annotations", VAL_ANNOTATIONS]
        )
    run("prepare_yolo.py", *arguments)
print("Dataset:", PREPARED / "data.yaml")

## 3. Optional D-FINE annotation and merge
Select S/M/L/X and a matching checkpoint. The script checks architecture compatibility, excludes human-labeled images and held-out cameras, and skips images with no accepted detections. Five-column YOLO labels are written separately from confidence/provenance records. Repeated merges do not duplicate images.


In [ ]:
if UNLABELED_IMAGES is not None:
    assert ANNOTATION_CHECKPOINT is not None and Path(ANNOTATION_CHECKPOINT).is_file()
    pseudo = WORK / "pseudo"
    if not (pseudo / "annotations.json").is_file():
        run(
            "generate_pseudo_labels.py",
            "--annotation-model",
            ANNOTATION_MODEL,
            "--checkpoint",
            ANNOTATION_CHECKPOINT,
            "--checkpoint-classes",
            ANNOTATION_CHECKPOINT_CLASSES,
            "--images",
            UNLABELED_IMAGES,
            "--data",
            PREPARED / "data.yaml",
            "--output",
            pseudo,
            "--confidence",
            ANNOTATION_CONFIDENCE,
            "--device",
            "cuda",
        )
    run("prepare_yolo.py", "--data", PREPARED / "data.yaml", "--add-pseudo", pseudo)
else:
    print("Using human annotations only")

## 4. Train the baseline
This is the only training stage. Interrupted baseline runs can resume from `last.pt`. Pruned checkpoints are rejected by the training entrypoint.


In [ ]:
baseline_run = RUNS / "baseline"
BASELINE = baseline_run / "weights/best.pt"
if not (baseline_run / "workflow.json").is_file():
    arguments = [
        "--data",
        PREPARED / "data.yaml",
        "--model",
        YOLO_CHECKPOINT,
        "--output",
        baseline_run,
        "--epochs",
        EPOCHS,
        "--image-size",
        IMAGE_SIZE,
        "--batch-size",
        BATCH_SIZE,
        "--seed",
        0,
        "--device",
        "0",
    ]
    if (baseline_run / "weights/last.pt").is_file():
        arguments += ["--resume", baseline_run / "weights/last.pt"]
    run("train_yolo.py", *arguments)
assert BASELINE.is_file()

## 5. Prune and validate directly
Each candidate starts from the original baseline. Internal bottleneck channels are removed physically; sensitive interfaces and detection outputs are preserved. Channels are rounded to the nearest multiple of eight, so actual reductions differ from requested percentages. No recovery training is performed. Each candidate is immediately reloaded and validated.


In [ ]:
CHECKPOINTS = {"baseline": BASELINE}
for percentage in [10, 20]:
    output = RUNS / f"pruned{percentage}"
    if not (output / "pruning.json").is_file():
        run(
            "prune_yolo.py",
            "--checkpoint",
            BASELINE,
            "--data",
            PREPARED / "data.yaml",
            "--output",
            output,
            "--prune-ratio",
            percentage / 100,
            "--image-size",
            IMAGE_SIZE,
            "--device",
            "0",
        )
    CHECKPOINTS[f"pruned{percentage}"] = output / "pruned.pt"

## 6. Export nine FP32 ONNX models
Each original/pruned checkpoint is exported at 512, 640, and 960 with static batch size 1. Exports include model metadata and are checked against PyTorch using random input and a real validation image.


In [ ]:
manifest = json.loads((PREPARED / "manifest.json").read_text())
sample = PREPARED / "images" / manifest["splits"]["val"][0]["file_name"]
MODELS = WORK / "models"
for name, checkpoint in CHECKPOINTS.items():
    for size in EXPORT_SIZES:
        model = MODELS / f"{name}_{size}.onnx"
        run(
            "export_yolo.py",
            "--checkpoint",
            checkpoint,
            "--image-size",
            size,
            "--output",
            model,
            "--test-image",
            sample,
        )
run(
    "infer_yolo.py",
    "--model",
    MODELS / f"baseline_{EXPORT_SIZES[0]}.onnx",
    "--input",
    sample,
    "--output",
    WORK / "onnx_smoke_test",
)

## 7. Download the local CPU bundle
The zip contains ONNX models, sidecars, inference/benchmark scripts, requirements, and local instructions. With `INCLUDE_VALIDATION_IN_BUNDLE=True`, held-out validation images are copied into the zip so benchmarking works without Kaggle mounts. Training images and D-FINE checkpoints are not included. Validation images can make the zip large.

Benchmark on your actual local CPU using `LOCAL.md`. Choose the fastest candidate within 2 AP points of the baseline; retaining the original model is a valid result.


In [ ]:
arguments = ["--models", MODELS, "--output", WORK.parent / "yolo_local.zip"]
if INCLUDE_VALIDATION_IN_BUNDLE:
    arguments += ["--data", PREPARED / "data.yaml"]
run("package_local.py", *arguments)
print("Download from notebook outputs:", WORK.parent / "yolo_local.zip")